# Early ICU mortality risk with temporal vitals

Compare gradient boosting with eight last-measured vitals against the same model with per-vital averages, variability, changes between time windows and observation counts. Report AUROC, average precision and Brier score on stratified 25% patient holdouts. Use 4,000 ICU stays from PhysioNet Challenge 2012 Set A, with measurements through ICU hour 24 and in-hospital mortality labels. This evaluates retrospective risk ranking, including the effect of measurement frequency. The three random splits can overlap.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'icu-temporal-warning-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scikit-learn==1.7.2', 'scipy==1.15.3', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import os\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\nroot = (\n    Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\n    / "icu2012"\n)\nroot.mkdir(parents=True, exist_ok=True)\nfor name in ["set-a.tar.gz", "Outcomes-a.txt"]:\n    target = root / name\n    if not target.exists():\n        temp = target.with_suffix(".part")\n        urlretrieve("https://physionet.org/files/challenge-2012/1.0.0/" + name, temp)\n        temp.replace(target)\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    brier_score_loss,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nimport csv\nimport io\nimport tarfile\n\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.metrics import brier_score_loss\nfrom sklearn.model_selection import train_test_split\n\nVITALS = ["HR", "RespRate", "Temp", "SysABP", "SpO2", "MAP", "NISysABP", "SaO2"]\n\n\ndef summarize(series):\n    static = []\n    dynamic = []\n    for name in VITALS:\n        values = sorted(series.get(name, []))\n        if values:\n            times, y = np.array(values).T\n            static.append(y[-1])\n            early = y[times < 12]\n            late = y[times >= 12]\n            change = late.mean() - early.mean() if len(early) and len(late) else np.nan\n            dynamic.extend([y.mean(), y.std(), change, np.log1p(len(y))])\n        else:\n            static.append(np.nan)\n            dynamic.extend([np.nan, np.nan, np.nan, 0])\n    return static + dynamic\n\n\ndef public():\n    folder = DATA / "icu2012"\n    with (folder / "Outcomes-a.txt").open() as stream:\n        labels = {\n            row["RecordID"]: int(row["In-hospital_death"])\n            for row in csv.DictReader(stream)\n        }\n    x = []\n    y = []\n    with tarfile.open(folder / "set-a.tar.gz") as archive:\n        for member in archive:\n            key = Path(member.name).stem\n            if key not in labels or not member.isfile():\n                continue\n            stream = io.TextIOWrapper(archive.extractfile(member))\n            series = {}\n            for row in csv.DictReader(stream):\n                if row["Parameter"] not in VITALS:\n                    continue\n                h, m = map(int, row["Time"].split(":"))\n                v = float(row["Value"])\n                if h + m / 60 <= 24 and v > 0:\n                    series.setdefault(row["Parameter"], []).append((h + m / 60, v))\n            x.append(summarize(series))\n            y.append(labels[key])\n    assert len(x) == 4000\n    return np.array(x), np.array(y)\n\n\ndef run(seed):\n    x, y = public()\n    train, test = train_test_split(\n        np.arange(len(y)), test_size=0.25, stratify=y, random_state=seed\n    )\n    out = {}\n    for method, k in [("last_values", 8), ("temporal_summaries", 40)]:\n        model = HistGradientBoostingClassifier(\n            max_iter=125,\n            min_samples_leaf=24,\n            max_leaf_nodes=15,\n            l2_regularization=2,\n            random_state=seed,\n        )\n        model.fit(x[train, :k], y[train])\n        p = model.predict_proba(x[test, :k])[:, 1]\n        out[method] = {\n            **scores(y[test], p),\n            "brier": float(brier_score_loss(y[test], p)),\n        }\n        save((model, test), f"{method}-{seed}")\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())